# LLM Serving: Continuous Batching & PagedAttention

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/llm/llm_serving_batching_paged_attention.ipynb)

Serving one request is a KV-cache problem. Serving hundreds at once is a scheduling and
memory-management problem. This notebook builds the two ideas that fixed it, as small
discrete-event simulators (not GPU kernels):

1. **Continuous batching** (Orca) vs static batching — GPU utilisation and latency.
2. **PagedAttention** (vLLM) vs contiguous KV allocation — memory and concurrency.

Companion post: *LLM Serving: Continuous Batching & PagedAttention* on sesen.ai.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb

## 1. The decode scheduler

Decoding runs one token per step for every sequence in the batch, and a decode step is
memory-bandwidth-bound, so its cost is ~constant regardless of batch size. We measure time
in decode steps. `grid_cols` optionally records a slot x step occupancy map for plotting.

In [ ]:
def make_workload(n, seed=0, short_frac=0.7, short=(8, 40), long=(200, 700)):
    rng = np.random.default_rng(seed)
    is_short = rng.random(n) < short_frac
    out_len = np.where(is_short, rng.integers(*short, n), rng.integers(*long, n))
    return out_len.astype(int), np.zeros(n, int)          # (output lengths, burst arrival)

def simulate_static(out_len, arrival, slots, grid_cols=0):
    """Run a full batch until its LONGEST request finishes, then start the next."""
    order = np.argsort(arrival, kind="stable")
    completion, active, t, i = np.zeros(len(out_len), int), [], 0, 0
    grid = -np.ones((slots, grid_cols), int) if grid_cols else None
    while i < len(out_len):
        if arrival[order[i]] > t: t = arrival[order[i]]
        batch = []
        while len(batch) < slots and i < len(out_len) and arrival[order[i]] <= t:
            batch.append(order[i]); i += 1
        rem = out_len[batch]
        for step in range(rem.max()):
            active.append(int((rem > step).sum()))
            if grid is not None and t + step < grid_cols:
                for j, r in enumerate(batch):
                    if out_len[r] > step: grid[j, t + step] = r
        for r in batch: completion[r] = t + out_len[r]
        t += rem.max()
    return {"completion": completion, "active": np.array(active), "slots": slots,
            "arrival": arrival, "out_len": out_len, "grid": grid}

def simulate_continuous(out_len, arrival, slots, grid_cols=0):
    """Every step: evict finished sequences, admit waiting ones into freed slots."""
    order = list(np.argsort(arrival, kind="stable"))
    completion = np.zeros(len(out_len), int)
    active, free, trace, nxt, t = {}, list(range(slots)), [], 0, 0
    grid = -np.ones((slots, grid_cols), int) if grid_cols else None
    while nxt < len(out_len) or active:
        while free and nxt < len(out_len) and arrival[order[nxt]] <= t:
            s = free.pop(0); active[s] = [order[nxt], out_len[order[nxt]]]; nxt += 1
        if not active: t = arrival[order[nxt]]; continue
        trace.append(len(active))
        if grid is not None and t < grid_cols:
            for s, a in active.items(): grid[s, t] = a[0]
        for s in list(active):
            active[s][1] -= 1
            if active[s][1] == 0:
                completion[active[s][0]] = t + 1; del active[s]; free.append(s)
        free.sort(); t += 1
    return {"completion": completion, "active": np.array(trace), "slots": slots,
            "arrival": arrival, "out_len": out_len, "grid": grid}

def metrics(r):
    lat = r["completion"] - r["arrival"]
    return {"utilisation": r["active"].mean() / r["slots"],
            "mean_latency": lat.mean(), "p95_latency": np.percentile(lat, 95),
            "makespan": len(r["active"]),
            "throughput": r["out_len"].sum() / len(r["active"])}

### Static vs continuous

Same 300 requests, same 32 slots. Continuous batching keeps the GPU full.

In [ ]:
out_len, arrival = make_workload(300)
stat = simulate_static(out_len, arrival, 32, grid_cols=260)
cont = simulate_continuous(out_len, arrival, 32, grid_cols=260)
ms, mc = metrics(stat), metrics(cont)
print(f"{'metric':<14}{'static':>12}{'continuous':>14}")
for k in ["utilisation", "mean_latency", "p95_latency", "makespan", "throughput"]:
    print(f"{k:<14}{ms[k]:>12.1f}{mc[k]:>14.1f}")
print(f"\nthroughput gain {mc['throughput']/ms['throughput']:.1f}x, "
      f"latency cut {ms['mean_latency']/mc['mean_latency']:.1f}x")

plt.figure(figsize=(8, 4))
plt.plot(stat["active"] / 32 * 100, label="static", color="#ea580c")
plt.plot(cont["active"] / 32 * 100, label="continuous", color="#2563eb")
plt.xlabel("decode step"); plt.ylabel("GPU utilisation (%)"); plt.legend()
plt.title("Continuous batching keeps the GPU full"); plt.show()

### Occupancy map

Each GPU slot over time. White is a wasted (idle) slot: static drains, continuous refills.

In [ ]:
PALETTE = ["#2563eb", "#16a34a", "#9333ea", "#0891b2", "#ca8a04", "#db2777",
           "#4f46e5", "#0d9488", "#7c3aed", "#c2410c", "#15803d", "#1d4ed8"]
PAL = np.array([to_rgb(c) for c in PALETTE])

def grid_rgb(grid):
    img = np.ones((*grid.shape, 3)); busy = grid >= 0
    img[busy] = PAL[grid[busy] % len(PALETTE)]; return img

fig, axes = plt.subplots(2, 1, figsize=(10, 4.5), sharex=True)
for ax, r, name in [(axes[0], stat, "static"), (axes[1], cont, "continuous")]:
    ax.imshow(grid_rgb(r["grid"]), aspect="auto", interpolation="nearest")
    ax.set_ylabel("slot"); ax.set_title(f"{name}  ({metrics(r)['utilisation']*100:.0f}% used)")
axes[1].set_xlabel("decode step"); plt.tight_layout(); plt.show()

## 2. KV memory: contiguous vs PagedAttention

Each sequence needs KV memory proportional to its (growing) length. Contiguous allocation
reserves a max-length slab per request; paging grabs fixed-size blocks on demand.

In [ ]:
def contiguous_capacity(seq_lens, max_len, capacity):
    n_fit = min(capacity // max_len, len(seq_lens))
    return n_fit, int(seq_lens[:n_fit].sum()) / capacity

def paged_capacity(seq_lens, capacity, block=16):
    total, used_blocks, used, n_fit = capacity // block, 0, 0, 0
    for L in seq_lens:
        need = int(np.ceil(L / block))
        if used_blocks + need > total: break
        used_blocks += need; used += int(L); n_fit += 1
    return n_fit, used / capacity

lens = np.random.default_rng(1).integers(20, 256, size=400)
cg_fit, cg_u = contiguous_capacity(lens, 512, 16384)
pg_fit, pg_u = paged_capacity(lens, 16384)
print(f"contiguous: {cg_fit:>3} requests, {cg_u*100:.0f}% memory used")
print(f"paged     : {pg_fit:>3} requests, {pg_u*100:.0f}% memory used")
print(f"-> {pg_fit/cg_fit:.1f}x more concurrent requests in the same memory")

plt.figure(figsize=(6, 4))
plt.bar(["contiguous", "paged"], [cg_fit, pg_fit], color=["#ea580c", "#16a34a"])
plt.ylabel("concurrent requests"); plt.title("Fragmentation vanishes"); plt.show()

## Exercises

1. **Arrival rate.** Give requests staggered Poisson arrivals instead of a burst. How does
   the utilisation gap between static and continuous change under light vs heavy load?
2. **Slot count.** Sweep `slots` from 4 to 128. Where does continuous batching stop helping
   (when the backlog no longer refills the batch)?
3. **Block size.** In `paged_capacity`, try block sizes 1, 8, 16, 64. What trades off between
   internal waste (large blocks) and bookkeeping (small blocks)?
4. **Prefix sharing.** Add copy-on-write: if two requests share the first k tokens, let them
   share those blocks. How much memory does a common system prompt save?
5. **Chunked prefill.** Model a long prompt's prefill as blocking decode for many steps, then
   slice it into chunks interleaved with decode. Watch tail latency improve.

### References

- Yu et al. (2022), *Orca: A Distributed Serving System...* (continuous batching)
- Kwon et al. (2023), *Efficient Memory Management for LLM Serving with PagedAttention* (vLLM), arXiv:2309.06180
- Pope et al. (2022), *Efficiently Scaling Transformer Inference*
- Agrawal et al. (2024), *Sarathi-Serve* (chunked prefill)